# Exploratory data analysis: Home Credit repayment difficulty

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),Path.cwd().parent] if (p / "data" / "raw").exists()); RAW=ROOT / "data" / "raw"
train=pd.read_csv(RAW / "application_train.csv")
test=pd.read_csv(RAW / "application_test.csv")
dictionary=pd.read_csv(RAW / "HomeCredit_columns_description.csv", encoding="latin1")
def show(x, sentence):
    display(x if isinstance(x, pd.DataFrame) else x.to_frame())
    print("What the output shows: " + sentence)
def rates(d, col):
    return d.groupby(col, dropna=False).TARGET.agg(["count", "sum", "mean"]).rename(columns={"sum":"difficulty_count", "mean":"difficulty_rate"}).sort_values("count", ascending=False)
print(f"Loaded {len(train):,} labeled and {len(test):,} unlabeled applications.")

Loaded 307,511 labeled and 48,744 unlabeled applications.


# Standing checks

## 1. Target distribution

In [2]:
x=rates(train,"TARGET"); x["share"]=x["count"]/len(train); show(x, "the target counts and shares in the labeled application population.")

,count,difficulty_count,difficulty_rate,share
TARGET,,,,
0,282686,0,0.0,0.919271
1,24825,24825,1.0,0.080729


What the output shows: the target counts and shares in the labeled application population.


## 2. Missingness and target relationship

In [3]:
x=pd.DataFrame({"overall":train.isna().mean(),"target_0":train[train.TARGET.eq(0)].isna().mean(),"target_1":train[train.TARGET.eq(1)].isna().mean()}); x["difference"]=x.target_1-x.target_0; show(x.reindex(x.difference.abs().sort_values(ascending=False).index).head(20), "the fields with the largest observed missingness differences between TARGET groups.")

,overall,target_0,target_1,difference
EMERGENCYSTATE_MODE,0.473983,0.467858,0.543726,0.075868
TOTALAREA_MODE,0.482685,0.476589,0.552105,0.075516
ENTRANCES_MODE,0.503488,0.497432,0.572447,0.075015
ENTRANCES_AVG,0.503488,0.497432,0.572447,0.075015
ENTRANCES_MEDI,0.503488,0.497432,0.572447,0.075015
FLOORSMAX_AVG,0.497608,0.491556,0.566526,0.074970
FLOORSMAX_MEDI,0.497608,0.491556,0.566526,0.074970
FLOORSMAX_MODE,0.497608,0.491556,0.566526,0.074970
YEARS_BEGINEXPLUATATION_AVG,0.487810,0.481803,0.556213,0.074410
YEARS_BEGINEXPLUATATION_MEDI,0.487810,0.481803,0.556213,0.074410


What the output shows: the fields with the largest observed missingness differences between TARGET groups.


## 3. Impossible values

In [4]:
d=[c for c in train if c.startswith("DAYS_")]; x=pd.DataFrame({"positive_count":[(train[c]>0).sum() for c in d],"sentinel_365243_count":[(train[c]==365243).sum() for c in d],"minimum":[train[c].min() for c in d],"maximum":[train[c].max() for c in d]},index=d); show(x.sort_values("sentinel_365243_count",ascending=False), "positive relative-day and 365243-sentinel counts in application day fields.")

,positive_count,sentinel_365243_count,minimum,maximum
DAYS_EMPLOYED,55374,55374,-17912.0,365243.0
DAYS_BIRTH,0,0,-25229.0,-7489.0
DAYS_REGISTRATION,0,0,-24672.0,0.0
DAYS_ID_PUBLISH,0,0,-7197.0,0.0
DAYS_LAST_PHONE_CHANGE,0,0,-4292.0,0.0


What the output shows: positive relative-day and 365243-sentinel counts in application day fields.


## 4. Keys, duplicates, and table grain

In [5]:
x=pd.DataFrame([[n,len(d),d.SK_ID_CURR.nunique(),d.duplicated().sum(),d.drop(columns="SK_ID_CURR").duplicated().sum()] for n,d in [("train",train),("test",test)]],columns=["split","rows","unique_SK_ID_CURR","duplicate_rows","duplicate_non_ID_rows"]); show(x,"application-key uniqueness and exact duplicate counts in each split.")

,split,rows,unique_SK_ID_CURR,duplicate_rows,duplicate_non_ID_rows
0,train,307511,307511,0,0
1,test,48744,48744,0,0


What the output shows: application-key uniqueness and exact duplicate counts in each split.


## 5. Temporal direction

In [6]:
x=dictionary[dictionary.Row.astype(str).str.contains("DAYS_|MONTHS_",regex=True,na=False)][["Table","Row","Description"]].copy(); x["status"]=np.where(x.Row.isin(["DAYS_BIRTH","DAYS_EMPLOYED","DAYS_REGISTRATION","DAYS_ID_PUBLISH","DAYS_LAST_PHONE_CHANGE"]),"application field; sign/sentinel review","historical or ambiguous; exclude pending as-of proof"); show(x,"the temporal dictionary fields and current application-time review statuses.")

,Table,Row,Description,status
17,application_{train|test}.csv,DAYS_BIRTH,Client's age in days at the time of application,application field; sign/sentinel review
18,application_{train|test}.csv,DAYS_EMPLOYED,How many days before the application the perso...,application field; sign/sentinel review
19,application_{train|test}.csv,DAYS_REGISTRATION,How many days before the application did clien...,application field; sign/sentinel review
20,application_{train|test}.csv,DAYS_ID_PUBLISH,How many days before the application did clien...,application field; sign/sentinel review
95,application_{train|test}.csv,DAYS_LAST_PHONE_CHANGE,How many days before application did client ch...,application field; sign/sentinel review
126,bureau.csv,DAYS_CREDIT,How many days before current application did c...,historical or ambiguous; exclude pending as-of...
128,bureau.csv,DAYS_CREDIT_ENDDATE,Remaining duration of CB credit (in days) at t...,historical or ambiguous; exclude pending as-of...
129,bureau.csv,DAYS_ENDDATE_FACT,Days since CB credit ended at the time of appl...,historical or ambiguous; exclude pending as-of...
137,bureau.csv,DAYS_CREDIT_UPDATE,How many days before loan application did last...,historical or ambiguous; exclude pending as-of...
140,bureau_balance.csv,MONTHS_BALANCE,Month of balance relative to application date ...,historical or ambiguous; exclude pending as-of...


What the output shows: the temporal dictionary fields and current application-time review statuses.


## 6. Train/test columns and category levels

In [7]:
s=sorted(set(train)-{"TARGET"}); x=pd.DataFrame({"train_dtype":train[s].dtypes.astype(str),"test_dtype":test[s].dtypes.astype(str),"train_missing":train[s].isna().mean(),"test_missing":test[s].isna().mean()});x["missingness_difference"]=x.train_missing-x.test_missing;show(x.reindex(x.missingness_difference.abs().sort_values(ascending=False).index).head(20),"data types and missingness differences for shared columns; TARGET is train-only.")

,train_dtype,test_dtype,train_missing,test_missing,missingness_difference
EXT_SOURCE_1,float64,float64,0.563811,0.421221,0.142590
EXT_SOURCE_3,float64,float64,0.198253,0.177827,0.020426
ENTRANCES_MODE,float64,float64,0.503488,0.483731,0.019756
ENTRANCES_MEDI,float64,float64,0.503488,0.483731,0.019756
ENTRANCES_AVG,float64,float64,0.503488,0.483731,0.019756
FLOORSMAX_MEDI,float64,float64,0.497608,0.478438,0.019170
FLOORSMAX_MODE,float64,float64,0.497608,0.478438,0.019170
FLOORSMAX_AVG,float64,float64,0.497608,0.478438,0.019170
YEARS_BEGINEXPLUATATION_MODE,float64,float64,0.487810,0.468899,0.018911
YEARS_BEGINEXPLUATATION_AVG,float64,float64,0.487810,0.468899,0.018911


What the output shows: data types and missingness differences for shared columns; TARGET is train-only.


# Questions in approved order

## 1. What is the payment-difficulty rate overall, and how imbalanced is the target?

In [8]:
x=rates(train,"TARGET");x["share"]=x["count"]/len(train);show(x,"the target counts, observed rates, and class imbalance.")

,count,difficulty_count,difficulty_rate,share
TARGET,,,,
0,282686,0,0.0,0.919271
1,24825,24825,1.0,0.080729


What the output shows: the target counts, observed rates, and class imbalance.


## 2. Are there duplicate application IDs or duplicate customer records?

In [9]:
x=pd.DataFrame([[n,d.SK_ID_CURR.duplicated().sum(),d.duplicated().sum(),d.drop(columns="SK_ID_CURR").duplicated().sum()] for n,d in [("train",train),("test",test)]],columns=["split","duplicate_IDs","duplicate_rows","duplicate_non_ID_rows"]);show(x,"duplicate counts for applications; no person key supports customer-level duplication.")

,split,duplicate_IDs,duplicate_rows,duplicate_non_ID_rows
0,train,0,0,0
1,test,0,0,0


What the output shows: duplicate counts for applications; no person key supports customer-level duplication.


## 3. Which variables have the most missing values, and does missingness differ between borrowers with and without payment difficulty?

In [10]:
x=pd.DataFrame({"overall":train.isna().mean(),"target_0":train[train.TARGET.eq(0)].isna().mean(),"target_1":train[train.TARGET.eq(1)].isna().mean()});x["absolute_difference"]=(x.target_1-x.target_0).abs();show(x.sort_values("absolute_difference",ascending=False).head(20),"the fields with the largest missingness difference by TARGET.")

,overall,target_0,target_1,absolute_difference
EMERGENCYSTATE_MODE,0.473983,0.467858,0.543726,0.075868
TOTALAREA_MODE,0.482685,0.476589,0.552105,0.075516
ENTRANCES_MODE,0.503488,0.497432,0.572447,0.075015
ENTRANCES_AVG,0.503488,0.497432,0.572447,0.075015
ENTRANCES_MEDI,0.503488,0.497432,0.572447,0.075015
FLOORSMAX_AVG,0.497608,0.491556,0.566526,0.074970
FLOORSMAX_MEDI,0.497608,0.491556,0.566526,0.074970
FLOORSMAX_MODE,0.497608,0.491556,0.566526,0.074970
YEARS_BEGINEXPLUATATION_AVG,0.487810,0.481803,0.556213,0.074410
YEARS_BEGINEXPLUATATION_MEDI,0.487810,0.481803,0.556213,0.074410


What the output shows: the fields with the largest missingness difference by TARGET.


## 4. Which fields use sentinel or impossible values (for example, 365243 in employment-related fields), and how should they be recoded?

In [11]:
x=pd.DataFrame([["DAYS_EMPLOYED","365243",int((train.DAYS_EMPLOYED==365243).sum()),"set missing and retain indicator","positive value conflicts with before-application definition"]],columns=["field","condition","count","proposed_not_applied_treatment","rationale"]);show(x,"the employment sentinel and a proposed, not applied, treatment.")

,field,condition,count,proposed_not_applied_treatment,rationale
0,DAYS_EMPLOYED,365243,55374,set missing and retain indicator,positive value conflicts with before-applicati...


What the output shows: the employment sentinel and a proposed, not applied, treatment.


## 5. Are there invalid, zero, negative, or extreme values in income, credit amount, annuity, goods price, and day-based variables?

In [12]:
c=["AMT_INCOME_TOTAL","AMT_CREDIT","AMT_ANNUITY","AMT_GOODS_PRICE","DAYS_BIRTH","DAYS_EMPLOYED","DAYS_REGISTRATION","DAYS_ID_PUBLISH"];x=train[c].describe(percentiles=[.01,.5,.99]).T;x["zero_count"]=(train[c]==0).sum();x["negative_count"]=(train[c]<0).sum();show(x,"ranges, selected quantiles, and zero/negative counts for the requested fields.")

,count,mean,std,min,1%,50%,99%,max,zero_count,negative_count
AMT_INCOME_TOTAL,307511.0,168797.919297,237123.146279,25650.0,45000.00,147150.0,472500.0,117000000.0,0,0
AMT_CREDIT,307511.0,599025.999706,402490.776996,45000.0,76410.00,513531.0,1854000.0,4050000.0,0,0
AMT_ANNUITY,307499.0,27108.573909,14493.737315,1615.5,6182.91,24903.0,70006.5,258025.5,0,0
AMT_GOODS_PRICE,307233.0,538396.207429,369446.460540,40500.0,67500.00,450000.0,1800000.0,4050000.0,0,0
DAYS_BIRTH,307511.0,-16036.995067,4363.988632,-25229.0,-24419.00,-15750.0,-8263.0,-7489.0,0,307511
DAYS_EMPLOYED,307511.0,63815.045904,141275.766519,-17912.0,-10894.90,-1213.0,365243.0,365243.0,2,252135
DAYS_REGISTRATION,307511.0,-4986.120328,3522.886321,-24672.0,-13879.00,-4504.0,-50.0,0.0,80,307431
DAYS_ID_PUBLISH,307511.0,-2994.202373,1509.450419,-7197.0,-5447.00,-3254.0,-61.0,0.0,16,307495


What the output shows: ranges, selected quantiles, and zero/negative counts for the requested fields.


## 33. Do train and test have the same feature columns, data types, missingness conventions, and supported category levels?

In [13]:
s=sorted(set(train)-{"TARGET"});rows=[]
for c in train.select_dtypes("object"):
 if c in test: rows.append([c,len(set(train[c].dropna())-set(test[c].dropna())),len(set(test[c].dropna())-set(train[c].dropna()))])
x=pd.DataFrame(rows,columns=["categorical_field","train_only_levels","test_only_levels"]);show(x,"train-only and test-only category-level counts for shared categorical fields.")

,categorical_field,train_only_levels,test_only_levels
0,NAME_CONTRACT_TYPE,0,0
1,CODE_GENDER,1,0
2,FLAG_OWN_CAR,0,0
3,FLAG_OWN_REALTY,0,0
4,NAME_TYPE_SUITE,0,0
5,NAME_INCOME_TYPE,1,0
6,NAME_EDUCATION_TYPE,0,0
7,NAME_FAMILY_STATUS,1,0
8,NAME_HOUSING_TYPE,0,0
9,OCCUPATION_TYPE,0,0


What the output shows: train-only and test-only category-level counts for shared categorical fields.


## 7. Are categorical values standardized, or are there rare, inconsistent, or unknown categories?

In [14]:
rows=[]
for c in train.select_dtypes("object"):
 s=train[c].fillna("<MISSING>");rows.append([c,s.nunique(),(s.value_counts()<10).sum(),(s=="<MISSING>").sum()])
x=pd.DataFrame(rows,columns=["field","levels","rare_levels_under_10","missing_rows"]);show(x.sort_values("rare_levels_under_10",ascending=False),"categorical level counts, rare-level counts, and missing rows.")

,field,levels,rare_levels_under_10,missing_rows
1,CODE_GENDER,3,1,0
5,NAME_INCOME_TYPE,8,1,0
7,NAME_FAMILY_STATUS,6,1,0
0,NAME_CONTRACT_TYPE,2,0,0
2,FLAG_OWN_CAR,2,0,0
3,FLAG_OWN_REALTY,2,0,0
4,NAME_TYPE_SUITE,8,0,1292
6,NAME_EDUCATION_TYPE,5,0,0
8,NAME_HOUSING_TYPE,6,0,0
9,OCCUPATION_TYPE,19,0,96391


What the output shows: categorical level counts, rare-level counts, and missing rows.


## 8. Does joining bureau, prior application, installment, card, and POS-cash tables create unintended duplicated records or inflated counts?

In [15]:
files=["bureau.csv","previous_application.csv","installments_payments.csv","credit_card_balance.csv","POS_CASH_balance.csv"];rows=[]
for f in files:
 d=pd.read_csv(RAW/f,usecols=["SK_ID_CURR"]);rows.append([f,len(d),d.SK_ID_CURR.nunique(),d.SK_ID_CURR.isin(pd.concat([train.SK_ID_CURR,test.SK_ID_CURR])).mean()])
x=pd.DataFrame(rows,columns=["table","rows","unique_applicants","key_resolves_to_application_union"]);show(x,"child-table rows and key resolution; raw child tables must be separately aggregated.")

,table,rows,unique_applicants,key_resolves_to_application_union
0,bureau.csv,1716428,305811,1.0
1,previous_application.csv,1670214,338857,1.0
2,installments_payments.csv,13605401,339587,1.0
3,credit_card_balance.csv,3840312,103558,1.0
4,POS_CASH_balance.csv,10001358,337252,1.0


What the output shows: child-table rows and key resolution; raw child tables must be separately aggregated.


## 28. Is every candidate feature genuinely available when the application is evaluated, rather than after loan issuance or repayment?

In [16]:
x=pd.DataFrame([["application fields","conditional","domain and sentinel review"],["bureau/previous","conditional","record-level date proof required"],["POS/card/installments","exclude pending proof","snapshot/payment timing not demonstrated"],["TARGET","exclude","outcome label"]],columns=["source","status","rule"]);show(x,"application-time availability decisions; historical features are not approved without an as-of rule.")

,source,status,rule
0,application fields,conditional,domain and sentinel review
1,bureau/previous,conditional,record-level date proof required
2,POS/card/installments,exclude pending proof,snapshot/payment timing not demonstrated
3,TARGET,exclude,outcome label


What the output shows: application-time availability decisions; historical features are not approved without an as-of rule.


## 29. Which variables could leak future information or encode the outcome too directly?

In [17]:
x=pd.DataFrame([["TARGET","exclude","outcome label"],["DAYS_ENTRY_PAYMENT and DAYS_INSTALMENT","exclude pending proof","payment timing"],["MONTHS_BALANCE","exclude pending proof","monthly snapshots"],["application inputs","conditional","source review"]],columns=["field_group","status","reason"]);show(x,"leakage statuses and reasons for high-risk field groups.")

,field_group,status,reason
0,TARGET,exclude,outcome label
1,DAYS_ENTRY_PAYMENT and DAYS_INSTALMENT,exclude pending proof,payment timing
2,MONTHS_BALANCE,exclude pending proof,monthly snapshots
3,application inputs,conditional,source review


What the output shows: leakage statuses and reasons for high-risk field groups.


## 6. How do distributions of income, credit amount, annuity, and credit-to-income ratio differ by target?

In [18]:
d=train.assign(CREDIT_TO_INCOME=train.AMT_CREDIT/train.AMT_INCOME_TOTAL);x=d.groupby("TARGET")[["AMT_INCOME_TOTAL","AMT_CREDIT","AMT_ANNUITY","CREDIT_TO_INCOME"]].quantile([.1,.5,.9]).unstack();show(x,"target-stratified 10th, 50th, and 90th percentiles for the requested measures.")

AMT_INCOME_TOTAL                     AMT_CREDIT                       \
                    0.1       0.5       0.9        0.1       0.5        0.9   
TARGET                                                                        
0               81000.0  148500.0  270000.0   180000.0  517788.0  1157670.0   
1               81000.0  135000.0  256500.0   199108.8  497520.0  1042560.0   

       AMT_ANNUITY                   CREDIT_TO_INCOME                      
               0.1      0.5      0.9              0.1       0.5       0.9  
TARGET                                                                     
0          11020.5  24876.0  46134.0         1.320667  3.266653  7.500000  
1          11835.0  25263.0  42642.0         1.400000  3.253143  7.142857

What the output shows: target-stratified 10th, 50th, and 90th percentiles for the requested measures.


## 9. Which applicant characteristics are most associated with payment difficulty: age, employment length, education, family status, housing type, or occupation?

In [19]:
d=train.assign(age_band=pd.cut(-train.DAYS_BIRTH/365.25,[20,30,40,50,60,100]));x=pd.concat([rates(d,"age_band"),rates(d,"NAME_EDUCATION_TYPE"),rates(d,"NAME_FAMILY_STATUS"),rates(d,"NAME_HOUSING_TYPE"),rates(d,"OCCUPATION_TYPE")]);show(x,"counts and observed target rates for requested applicant-characteristic groups.")

/var/folders/_s/pw5c6cw96gngn5qnxq2r7l_r0000gn/T/ipykernel_73856/1273762696.py:13: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  return d.groupby(col, dropna=False).TARGET.agg(["count", "sum", "mean"]).rename(columns={"sum":"difficulty_count", "mean":"difficulty_rate"}).sort_values("count", ascending=False)


,count,difficulty_count,difficulty_rate
"(30, 40]",82349,7897,0.095897
"(40, 50]",76581,5853,0.076429
"(50, 60]",68109,4168,0.061196
"(20, 30]",45186,5171,0.114438
"(60, 100]",35286,1736,0.049198
Secondary / secondary special,218391,19524,0.089399
Higher education,74863,4009,0.053551
Incomplete higher,10277,872,0.084850
Lower secondary,3816,417,0.109277
Academic degree,164,3,0.018293


What the output shows: counts and observed target rates for requested applicant-characteristic groups.


## 10. How does payment difficulty vary by income type and employment status?

In [20]:
d=train.assign(employment_status=np.where(train.DAYS_EMPLOYED.eq(365243),"sentinel",np.where(train.DAYS_EMPLOYED.eq(0),"zero","negative_relative_day")));x=d.groupby(["NAME_INCOME_TYPE","employment_status"]).TARGET.agg(["count","mean"]);show(x,"counts and observed target rates by income type and employment-status rule.")

count      mean
NAME_INCOME_TYPE     employment_status                      
Businessman          negative_relative_day      10  0.000000
Commercial associate negative_relative_day   71617  0.074843
Maternity leave      negative_relative_day       5  0.400000
Pensioner            negative_relative_day      10  0.000000
                     sentinel                55352  0.053873
State servant        negative_relative_day   21703  0.057550
Student              negative_relative_day      18  0.000000
Unemployed           sentinel                   22  0.363636
Working              negative_relative_day  158772  0.095880
                     zero                        2  0.500000

What the output shows: counts and observed target rates by income type and employment-status rule.


## Approved historical-linkage and timing rule

In [21]:
# Official Kaggle linkage and conservative pre-application filter supplied for this analysis.
# SK_ID_CURR links each history table to the current application; bureau_balance links through bureau on SK_ID_BUREAU.
# DAYS_CREDIT, MONTHS_BALANCE, DAYS_DECISION, DAYS_INSTALMENT, and DAYS_ENTRY_PAYMENT must be < 0.
# Values of 0 and all positive values are excluded because they are not clearly earlier than the current application.
bureau_raw=pd.read_csv(RAW / "bureau.csv")
bureau_pre=bureau_raw[bureau_raw.DAYS_CREDIT.lt(0)].copy()
prev_raw=pd.read_csv(RAW / "previous_application.csv")
prev_pre=prev_raw[prev_raw.DAYS_DECISION.lt(0)].copy()
parts=[]
for chunk in pd.read_csv(RAW / "bureau_balance.csv",usecols=['SK_ID_BUREAU','MONTHS_BALANCE','STATUS'],chunksize=1000000):
    x=chunk[chunk.MONTHS_BALANCE.lt(0)].assign(severe_dpd=chunk.STATUS.isin(['3','4','5']))
    parts.append(x.groupby('SK_ID_BUREAU').severe_dpd.max())
bb_status=pd.concat(parts).groupby(level=0).max()
bb_link=bureau_pre[['SK_ID_BUREAU','SK_ID_CURR']].merge(bb_status.rename('bureau_balance_severe_dpd'),left_on='SK_ID_BUREAU',right_index=True,how='left')
print(f"Retained {len(bureau_pre):,}/{len(bureau_raw):,} bureau rows with DAYS_CREDIT < 0 and {len(prev_pre):,}/{len(prev_raw):,} previous applications with DAYS_DECISION < 0.")
print(f"Linked {bb_link.bureau_balance_severe_dpd.notna().sum():,} date-filtered bureau_balance records through SK_ID_BUREAU.")
print("What the output shows: the documented linkage and conservative timing filters used for all subsequent historical summaries.")

Retained 1,716,403/1,716,428 bureau rows with DAYS_CREDIT < 0 and 1,670,214/1,670,214 previous applications with DAYS_DECISION < 0.
Linked 769,170 date-filtered bureau_balance records through SK_ID_BUREAU.
What the output shows: the documented linkage and conservative timing filters used for all subsequent historical summaries.


## 11. Do first-time borrowers differ from applicants with prior Home Credit or bureau history?

In [22]:
b=set(bureau_pre.SK_ID_CURR.unique());p=set(prev_pre.SK_ID_CURR.unique());d=train.assign(bureau_history=train.SK_ID_CURR.isin(b),prior_hc_history=train.SK_ID_CURR.isin(p));x=d.groupby(['bureau_history','prior_hc_history']).TARGET.agg(['count','mean']);show(x,'the date-filtered bureau and prior-Home-Credit coverage groups with their observed target rates.')

count      mean
bureau_history prior_hc_history                  
False          False               2470  0.072874
               True               41551  0.102934
True           False              13984  0.057208
               True              249506  0.078427

What the output shows: the date-filtered bureau and prior-Home-Credit coverage groups with their observed target rates.


## 12. Is payment difficulty higher for applicants with larger loans, longer repayment obligations, or higher annuity-to-income ratios?

In [23]:
d=train.assign(credit_band=pd.qcut(train.AMT_CREDIT,5,duplicates='drop'),annuity_income_band=pd.qcut(train.AMT_ANNUITY/train.AMT_INCOME_TOTAL,5,duplicates='drop'));x=pd.concat([rates(d,'credit_band'),rates(d,'annuity_income_band')]);show(x,'the count and observed target-rate bins for credit amount and annuity-to-income; current CNT_PAYMENT is absent.')

/var/folders/_s/pw5c6cw96gngn5qnxq2r7l_r0000gn/T/ipykernel_73856/1273762696.py:13: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  return d.groupby(col, dropna=False).TARGET.agg(["count", "sum", "mean"]).rename(columns={"sum":"difficulty_count", "mean":"difficulty_rate"}).sort_values("count", ascending=False)


,count,difficulty_count,difficulty_rate
"(44999.999, 254700.0]",64925,4699,0.072376
"(604152.0, 900000.0]",64024,5029,0.078549
"(432000.0, 604152.0]",61552,6189,0.100549
"(900000.0, 4050000.0]",58912,3579,0.060752
"(254700.0, 432000.0]",58098,5329,0.091724
"(0.186, 0.247]",61522,5350,0.086961
"(0.104, 0.144]",61507,4817,0.078316
"(-0.000776, 0.104]",61500,4426,0.071967
"(0.144, 0.186]",61494,4997,0.081260
"(0.247, 1.876]",61476,5235,0.085155


What the output shows: the count and observed target-rate bins for credit amount and annuity-to-income; current CNT_PAYMENT is absent.


## 13. Which combinations of loan amount, income, household size, and employment history appear most financially stressed?

In [24]:
d=train.assign(loan_income=pd.qcut(train.AMT_CREDIT/train.AMT_INCOME_TOTAL,4,duplicates='drop'),family_size=pd.cut(train.CNT_FAM_MEMBERS,[0,1,2,4,100]));x=d.groupby(['loan_income','family_size'],observed=False).TARGET.agg(['count','mean']);show(x,'the pre-specified loan-to-income by family-size grid with counts and observed target rates.')

count      mean
loan_income      family_size                 
(0.00381, 2.019] (0, 1]       20303  0.077082
                 (1, 2]       35646  0.072491
                 (2, 4]       20025  0.075406
                 (4, 100]       973  0.085303
(2.019, 3.265]   (0, 1]       18786  0.090120
                 (1, 2]       37832  0.081941
                 (2, 4]       19238  0.094345
                 (4, 100]       961  0.095734
(3.265, 5.16]    (0, 1]       16567  0.088006
                 (1, 2]       40331  0.082368
                 (2, 4]       18973  0.096664
                 (4, 100]       997  0.107322
(5.16, 84.737]   (0, 1]       12191  0.078665
                 (1, 2]       44548  0.067410
                 (2, 4]       19062  0.083150
                 (4, 100]      1076  0.106877

What the output shows: the pre-specified loan-to-income by family-size grid with counts and observed target rates.


## 14. Are there geographic or organization-type patterns in application volume or payment difficulty?

In [25]:
x=pd.concat([rates(train,'REGION_RATING_CLIENT'),rates(train,'ORGANIZATION_TYPE')]);show(x,'the available region-rating and organization-type application counts and observed target rates; no geographic code supports a map.')

,count,difficulty_count,difficulty_rate
2,226984,17907,0.078891
3,48330,5366,0.111028
1,32197,1552,0.048203
Business Entity Type 3,67992,6323,0.092996
XNA,55374,2990,0.053996
...,...,...,...
Religion,85,5,0.058824
Industry: type 13,67,9,0.134328
Trade: type 4,64,2,0.031250
Trade: type 5,49,3,0.061224


What the output shows: the available region-rating and organization-type application counts and observed target rates; no geographic code supports a map.


## 15. Do document-submission and contact-information flags indicate application quality, fraud risk, or merely process differences?

In [26]:
f=[c for c in train if c.startswith('FLAG_DOCUMENT_') or c in ['FLAG_MOBIL','FLAG_EMP_PHONE','FLAG_WORK_PHONE','FLAG_CONT_MOBILE','FLAG_PHONE','FLAG_EMAIL']];x=pd.DataFrame([[c,int(train[c].sum()),train.groupby(c).TARGET.mean().to_dict()] for c in f],columns=['flag','positive_count','target_rate_by_flag']);show(x.sort_values('positive_count',ascending=False),'flag prevalence and observed target rates; no fraud label is present.')

,flag,positive_count,target_rate_by_flag
0,FLAG_MOBIL,307510,"{0: 0.0, 1: 0.08072908198107379}"
3,FLAG_CONT_MOBILE,306937,"{0: 0.078397212543554, 1: 0.08073317977304789}"
1,FLAG_EMP_PHONE,252125,"{0: 0.05400281659625176, 1: 0.08659990084283589}"
7,FLAG_DOCUMENT_3,218340,"{0: 0.06182503280214419, 1: 0.08844920765778144}"
4,FLAG_PHONE,86431,"{0: 0.08478378867378325, 1: 0.07035670072080619}"
2,FLAG_WORK_PHONE,61308,"{0: 0.07685121627275053, 1: 0.09630064591896653}"
10,FLAG_DOCUMENT_6,27078,"{0: 0.08314998591463915, 1: 0.05565403648718517}"
12,FLAG_DOCUMENT_8,25024,"{0: 0.08138073610467031, 1: 0.07336956521739131}"
5,FLAG_EMAIL,17442,"{0: 0.08084628140201125, 1: 0.07877536979704163}"
9,FLAG_DOCUMENT_5,4648,"{0: 0.08073947626484582, 1: 0.08003442340791739}"


What the output shows: flag prevalence and observed target rates; no fraud label is present.


## 16. How does payment difficulty vary by number of active, closed, overdue, or written-off bureau accounts?

In [27]:
b=bureau_pre.assign(overdue=(bureau_pre.CREDIT_DAY_OVERDUE.gt(0)|bureau_pre.AMT_CREDIT_SUM_OVERDUE.gt(0))).groupby('SK_ID_CURR').agg(accounts=('SK_ID_BUREAU','size'),active=('CREDIT_ACTIVE',lambda s:(s=='Active').sum()),closed=('CREDIT_ACTIVE',lambda s:(s=='Closed').sum()),overdue=('overdue','sum'));bb=bb_link.groupby('SK_ID_CURR').bureau_balance_severe_dpd.max().rename('severe_dpd');d=train[['SK_ID_CURR','TARGET']].merge(b,left_on='SK_ID_CURR',right_index=True,how='left').merge(bb,left_on='SK_ID_CURR',right_index=True,how='left').fillna(0);x=rates(d.assign(account_band=pd.cut(d.accounts,[-1,0,1,3,10,1e9])),'account_band');show(x,'date-filtered bureau-account count bands and observed target rates after applicant-level aggregation; the linked monthly status is retained as severe_dpd and no direct written-off code is assumed.')

/var/folders/_s/pw5c6cw96gngn5qnxq2r7l_r0000gn/T/ipykernel_73856/1273762696.py:13: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  return d.groupby(col, dropna=False).TARGET.agg(["count", "sum", "mean"]).rename(columns={"sum":"difficulty_count", "mean":"difficulty_rate"}).sort_values("count", ascending=False)


,count,difficulty_count,difficulty_rate
account_band,,,
"(3.0, 10.0]",126582,9340,0.073786
"(1.0, 3.0]",68560,5289,0.077144
"(-1.0, 0.0]",44021,4457,0.101247
"(0.0, 1.0]",36071,3098,0.085886
"(10.0, 1000000000.0]",32277,2641,0.081823


What the output shows: date-filtered bureau-account count bands and observed target rates after applicant-level aggregation; the linked monthly status is retained as severe_dpd and no direct written-off code is assumed.


## 17. Are outstanding debt, overdue balances, prior credit utilization, and days past due predictive of payment difficulty?

In [28]:
b=bureau_pre.groupby('SK_ID_CURR')[['AMT_CREDIT_SUM_DEBT','AMT_CREDIT_SUM_LIMIT','AMT_CREDIT_SUM_OVERDUE','CREDIT_DAY_OVERDUE']].sum();b['debt_to_limit']=b.AMT_CREDIT_SUM_DEBT/b.AMT_CREDIT_SUM_LIMIT.replace(0,np.nan);cc=pd.read_csv(RAW/'credit_card_balance.csv',usecols=['SK_ID_CURR','MONTHS_BALANCE','AMT_BALANCE','AMT_CREDIT_LIMIT_ACTUAL','SK_DPD']);cc=cc[cc.MONTHS_BALANCE.lt(0)];ca=cc.groupby('SK_ID_CURR')[['AMT_BALANCE','AMT_CREDIT_LIMIT_ACTUAL','SK_DPD']].sum();ca['card_utilization']=ca.AMT_BALANCE/ca.AMT_CREDIT_LIMIT_ACTUAL.replace(0,np.nan);d=train[['SK_ID_CURR','TARGET']].merge(b,left_on='SK_ID_CURR',right_index=True,how='left').merge(ca[['card_utilization','SK_DPD']],left_on='SK_ID_CURR',right_index=True,how='left');x=d.groupby('TARGET')[['AMT_CREDIT_SUM_DEBT','AMT_CREDIT_SUM_OVERDUE','CREDIT_DAY_OVERDUE','debt_to_limit','card_utilization','SK_DPD']].median();show(x,'target-stratified medians for date-filtered bureau debt and same-snapshot credit-card balance/limit aggregates.')

,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_OVERDUE,CREDIT_DAY_OVERDUE,debt_to_limit,card_utilization,SK_DPD
TARGET,,,,,,
0,162063.0,0.0,0.0,6.397341,0.236291,0.0
1,256288.5,0.0,0.0,19.026014,0.497780,0.0


What the output shows: target-stratified medians for date-filtered bureau debt and same-snapshot credit-card balance/limit aggregates.


## 18. How do borrowers with no bureau history compare with borrowers who have established credit histories?

In [29]:
ids=set(bureau_pre.SK_ID_CURR.unique());d=train.assign(bureau_history=np.where(train.SK_ID_CURR.isin(ids),'observed date-filtered bureau record','no observed date-filtered bureau record'));x=d.groupby('bureau_history').agg(applications=('TARGET','size'),target_rate=('TARGET','mean'),median_income=('AMT_INCOME_TOTAL','median'));show(x,'application counts, observed target rates, and median income by date-filtered bureau-record coverage.')

,applications,target_rate,median_income
bureau_history,,,
no observed date-filtered bureau record,44021,0.101247,135000.0
observed date-filtered bureau record,263490,0.077301,157500.0


What the output shows: application counts, observed target rates, and median income by date-filtered bureau-record coverage.


## 19. Does previous application behavior—approved, refused, canceled, or unused offers—relate to current payment difficulty?

In [30]:
a=prev_pre.pivot_table(index='SK_ID_CURR',columns='NAME_CONTRACT_STATUS',values='SK_ID_PREV',aggfunc='size',fill_value=0);d=train[['SK_ID_CURR','TARGET']].merge(a,left_on='SK_ID_CURR',right_index=True,how='left').fillna(0);x=d.groupby(pd.cut(d.get('Approved',pd.Series(0,index=d.index)),[-1,0,1,3,10000])).TARGET.agg(['count','mean']);show(x,'target rates by count of date-filtered prior applications with Approved status.')

/var/folders/_s/pw5c6cw96gngn5qnxq2r7l_r0000gn/T/ipykernel_73856/1756984057.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  a=prev_pre.pivot_table(index='SK_ID_CURR',columns='NAME_CONTRACT_STATUS',values='SK_ID_PREV',aggfunc='size',fill_value=0);d=train[['SK_ID_CURR','TARGET']].merge(a,left_on='SK_ID_CURR',right_index=True,how='left').fillna(0);x=d.groupby(pd.cut(d.get('Approved',pd.Series(0,index=d.index)),[-1,0,1,3,10000])).TARGET.agg(['count','mean']);show(x,'target rates by count of date-filtered prior applications with Approved status.')


,count,mean
Approved,,
"(-1, 0]",17446,0.061103
"(0, 1]",76640,0.093776
"(1, 3]",119626,0.083845
"(3, 10000]",93799,0.069745


What the output shows: target rates by count of date-filtered prior applications with Approved status.


## 20. Are repeat borrowers with successful installment histories less risky than comparable first-time borrowers?

In [31]:
inst=pd.read_csv(RAW/'installments_payments.csv',usecols=['SK_ID_CURR','DAYS_INSTALMENT','DAYS_ENTRY_PAYMENT','AMT_INSTALMENT','AMT_PAYMENT']);inst=inst[inst.DAYS_INSTALMENT.lt(0)&inst.DAYS_ENTRY_PAYMENT.lt(0)];a=inst.assign(on_or_above=inst.AMT_PAYMENT.ge(inst.AMT_INSTALMENT)).groupby('SK_ID_CURR').agg(prior_installments=('on_or_above','size'),on_or_above_share=('on_or_above','mean'));d=train[['SK_ID_CURR','TARGET']].merge(a,left_on='SK_ID_CURR',right_index=True,how='left');d['history_group']=np.where(d.prior_installments.isna(),'no qualifying installment history',np.where(d.on_or_above_share.eq(1),'all qualifying payments at/above installment','some qualifying payment below installment'));show(rates(d,'history_group'),'the date-filtered installment-history groups and their observed target rates, without a causal or matched comparison claim.')

,count,difficulty_count,difficulty_rate
history_group,,,
all qualifying payments at/above installment,166489,11826,0.071032
some qualifying payment below installment,125146,12049,0.096280
no qualifying installment history,15876,950,0.059839


What the output shows: the date-filtered installment-history groups and their observed target rates, without a causal or matched comparison claim.


## 21. Which prior-loan features matter most: product type, contract status, repayment duration, prior delinquency, or payment timeliness?

In [32]:
x=prev_pre.groupby('SK_ID_CURR').agg(prior_applications=('SK_ID_PREV','size'),median_prior_terms=('CNT_PAYMENT','median'),latest_decision=('DAYS_DECISION','max'));d=train[['SK_ID_CURR','TARGET']].merge(x,left_on='SK_ID_CURR',right_index=True,how='left');show(d.groupby('TARGET')[['prior_applications','median_prior_terms','latest_decision']].median(),'target-stratified medians for date-filtered previous-application aggregates; this is not a feature-importance ranking.')

,prior_applications,median_prior_terms,latest_decision
TARGET,,,
0,4.0,12.0,-302.0
1,4.0,12.0,-272.0


What the output shows: target-stratified medians for date-filtered previous-application aggregates; this is not a feature-importance ranking.


## 22. Where do bureau data and prior Home Credit data disagree about a borrower’s repayment history?

In [33]:
b=bureau_pre.assign(bureau_overdue=(bureau_pre.CREDIT_DAY_OVERDUE.gt(0)|bureau_pre.AMT_CREDIT_SUM_OVERDUE.gt(0))).groupby('SK_ID_CURR').bureau_overdue.max();p=prev_pre.groupby('SK_ID_CURR').NAME_CONTRACT_STATUS.apply(lambda s:(s=='Refused').any());d=pd.DataFrame({'bureau_overdue':b,'prior_refused':p}).dropna();x=pd.crosstab(d.bureau_overdue,d.prior_refused);show(x,'the overlap table for non-harmonized bureau overdue and prior refusal constructs; it is not an error comparison.')

prior_refused,False,True
bureau_overdue,,
False,184096,103081
True,2202,1710


What the output shows: the overlap table for non-harmonized bureau overdue and prior refusal constructs; it is not an error comparison.


## 23. Are there meaningful payment-difficulty gaps across demographic groups such as age bands, gender, family status, income type, and housing type?

In [34]:
d=train.assign(age_band=pd.cut(-train.DAYS_BIRTH/365.25,[20,30,40,50,60,100]));x=pd.concat([rates(d,'age_band'),rates(d,'CODE_GENDER'),rates(d,'NAME_FAMILY_STATUS'),rates(d,'NAME_INCOME_TYPE'),rates(d,'NAME_HOUSING_TYPE')]);show(x,'group counts and observed payment-difficulty rates for the listed demographic fields.')

/var/folders/_s/pw5c6cw96gngn5qnxq2r7l_r0000gn/T/ipykernel_73856/1273762696.py:13: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  return d.groupby(col, dropna=False).TARGET.agg(["count", "sum", "mean"]).rename(columns={"sum":"difficulty_count", "mean":"difficulty_rate"}).sort_values("count", ascending=False)


,count,difficulty_count,difficulty_rate
"(30, 40]",82349,7897,0.095897
"(40, 50]",76581,5853,0.076429
"(50, 60]",68109,4168,0.061196
"(20, 30]",45186,5171,0.114438
"(60, 100]",35286,1736,0.049198
F,202448,14170,0.069993
M,105059,10655,0.101419
XNA,4,0,0.000000
Married,196432,14850,0.075599
Single / not married,45444,4457,0.098077


What the output shows: group counts and observed payment-difficulty rates for the listed demographic fields.


## 24. Do apparent risk differences persist after comparing applicants with similar loan sizes, income, and credit-history profiles?

In [35]:
ids=set(bureau_pre.SK_ID_CURR.unique());d=train.assign(credit_band=pd.qcut(train.AMT_CREDIT,4,duplicates='drop'),income_band=pd.qcut(train.AMT_INCOME_TOTAL,4,duplicates='drop'),history=np.where(train.SK_ID_CURR.isin(ids),'bureau record','no bureau record'));x=d.groupby(['CODE_GENDER','credit_band','income_band','history'],observed=False).TARGET.agg(['count','mean']);show(x[x['count']>=30],'descriptive target rates in loan, income, and date-filtered history strata with at least 30 records.')

count  \
CODE_GENDER credit_band           income_band             history                   
F           (44999.999, 270000.0] (25649.999, 112500.0]   bureau record     26727   
                                                          no bureau record   5742   
                                  (112500.0, 147150.0]    bureau record      7573   
                                                          no bureau record   1163   
                                  (147150.0, 202500.0]    bureau record      7143   
...                                                                           ...   
M           (808650.0, 4050000.0] (112500.0, 147150.0]    no bureau record    425   
                                  (147150.0, 202500.0]    bureau record      6978   
                                                          no bureau record   1054   
                                  (202500.0, 117000000.0] bureau record     12689   
                                                          no bureau record   1598   

                                                                                mean  
CODE_GENDER credit_band           income_band             history                     
F           (44999.999, 270000.0] (25649.999, 112500.0]   bureau record     0.059902  
                                                          no bureau record  0.081853  
                                  (112500.0, 147150.0]    bureau record     0.055328  
                                                          no bureau record  0.081685  
                                  (147150.0, 202500.0]    bureau record     0.058659  
...                                                                              ...  
M           (808650.0, 4050000.0] (112500.0, 147150.0]    no bureau record  0.112941  
                                  (147150.0, 202500.0]    bureau record     0.082832  
                                                          no bureau record  0.093928  
                                  (202500.0, 117000000.0] bureau record     0.060131  
                                                          no bureau record  0.066333  

[64 rows x 2 columns]

What the output shows: descriptive target rates in loan, income, and date-filtered history strata with at least 30 records.


## 25. Which features may act as socioeconomic proxies and therefore need special scrutiny before modeling?

In [36]:
x=pd.DataFrame([['REGION_RATING_CLIENT','geographic/economic context'],['NAME_HOUSING_TYPE','housing access'],['OCCUPATION_TYPE','employment/socioeconomic status'],['ORGANIZATION_TYPE','employment context'],['contact/document flags','process/device access'],['EXT_SOURCE_*','unexplained external source']],columns=['feature_group','plausible_proxy_pathway']);x['required_treatment']='governance review before modeling';show(x,'the governance-review inventory of candidate socioeconomic proxy fields.')

,feature_group,plausible_proxy_pathway,required_treatment
0,REGION_RATING_CLIENT,geographic/economic context,governance review before modeling
1,NAME_HOUSING_TYPE,housing access,governance review before modeling
2,OCCUPATION_TYPE,employment/socioeconomic status,governance review before modeling
3,ORGANIZATION_TYPE,employment context,governance review before modeling
4,contact/document flags,process/device access,governance review before modeling
5,EXT_SOURCE_*,unexplained external source,governance review before modeling


What the output shows: the governance-review inventory of candidate socioeconomic proxy fields.


## 26. Are missing-data patterns or thin credit files concentrated in particular demographic groups?

In [37]:
ids=set(bureau_pre.SK_ID_CURR.unique());d=train.assign(no_bureau=~train.SK_ID_CURR.isin(ids),occupation_missing=train.OCCUPATION_TYPE.isna(),age_band=pd.cut(-train.DAYS_BIRTH/365.25,[20,30,40,50,60,100]));x=d.groupby(['CODE_GENDER','age_band'],observed=False).agg(applications=('TARGET','size'),no_bureau_share=('no_bureau','mean'),occupation_missing_share=('occupation_missing','mean'));show(x,'group counts and date-filtered bureau-absence and occupation-missingness shares by gender and age band.')

applications  no_bureau_share  occupation_missing_share
CODE_GENDER age_band                                                          
F           (20, 30]          25649         0.205076                  0.184218
            (30, 40]          50133         0.128119                  0.174077
            (40, 50]          49622         0.117891                  0.188848
            (50, 60]          49361         0.126031                  0.524888
            (60, 100]         27683         0.139436                  0.882636
M           (20, 30]          19535         0.194113                  0.155925
            (30, 40]          32215         0.134161                  0.149030
            (40, 50]          26959         0.143922                  0.158537
            (50, 60]          18747         0.163013                  0.264202
            (60, 100]          7603         0.178482                  0.808760
XNA         (20, 30]              2         0.000000                  0.500000
            (30, 40]              1         0.000000                  0.000000
            (40, 50]              0              NaN                       NaN
            (50, 60]              1         0.000000                  1.000000
            (60, 100]             0              NaN                       NaN

What the output shows: group counts and date-filtered bureau-absence and occupation-missingness shares by gender and age band.


## 27. Are there applicant groups that appear underserved but show repayment outcomes comparable to the overall portfolio?

In [38]:
ids=set(bureau_pre.SK_ID_CURR.unique());d=train.assign(group=np.where(train.SK_ID_CURR.isin(ids),'observed date-filtered bureau record','no observed date-filtered bureau record'));x=d.groupby('group').TARGET.agg(['count','mean']);x.loc['overall']=[len(d),d.TARGET.mean()];show(x,'the conditional coverage-group comparison and overall target rate; it does not identify underserved access.')

,count,mean
group,,
no observed date-filtered bureau record,44021.0,0.101247
observed date-filtered bureau record,263490.0,0.077301
overall,307511.0,0.080729


What the output shows: the conditional coverage-group comparison and overall target rate; it does not identify underserved access.


## 30. Is there a usable time sequence to test whether distributions or outcomes drift across cohorts?

In [39]:
x=pd.DataFrame([['absolute current-application date','not present in documented schema'],['WEEKDAY_APPR_PROCESS_START / HOUR_APPR_PROCESS_START','not a calendar cohort key'],['relative dates','not an absolute cohort key']],columns=['candidate','decision']);show(x,'the temporal-availability decision: calendar-cohort drift is not answerable from the documented schema.')

,candidate,decision
0,absolute current-application date,not present in documented schema
1,WEEKDAY_APPR_PROCESS_START / HOUR_APPR_PROCESS...,not a calendar cohort key
2,relative dates,not an absolute cohort key


What the output shows: the temporal-availability decision: calendar-cohort drift is not answerable from the documented schema.


## 31. Because the data contain only issued loans, what questions about rejected applicants, approval rates, and policy effects cannot be answered from this sample?

In [40]:
x=pd.DataFrame([['approval/denial rate','not identifiable','no applicant/reject denominator'],['rejected-applicant repayment','not identifiable','no rejected outcomes'],['policy causal effect','not identifiable','issued-loan observational sample']],columns=['question','status','missing evidence']);show(x,'the issued-loan selection limitations and missing evidence for each question.')

,question,status,missing evidence
0,approval/denial rate,not identifiable,no applicant/reject denominator
1,rejected-applicant repayment,not identifiable,no rejected outcomes
2,policy causal effect,not identifiable,issued-loan observational sample


What the output shows: the issued-loan selection limitations and missing evidence for each question.


## 32. What additional current data would be needed to assess financial value—such as pricing, funding cost, loss given default, collections cost, and current policy thresholds?

In [41]:
x=pd.DataFrame([['pricing/revenue','loan-level yield and fees','Finance'],['funding cost','dated cost of funds','Finance'],['loss given default','recoveries, balances, charge-offs','Risk/Collections'],['collections cost','activity-level cost','Collections'],['policy thresholds','decision and override logs','Credit Risk']],columns=['business input','required data','source owner']);show(x,'the absent financial-value inputs and proposed source owner.')

,business input,required data,source owner
0,pricing/revenue,loan-level yield and fees,Finance
1,funding cost,dated cost of funds,Finance
2,loss given default,"recoveries, balances, charge-offs",Risk/Collections
3,collections cost,activity-level cost,Collections
4,policy thresholds,decision and override logs,Credit Risk


What the output shows: the absent financial-value inputs and proposed source owner.


# Final completeness check: what may be missed?

In [42]:
x=pd.DataFrame([
['Business Problem Statement','held-out AUC, calibration, selection and risk scenarios','required in validation, not EDA'],
['Business Problem Statement','financial contribution and adverse sensitivity','requires Finance inputs'],
['Dictionary','record-level temporal provenance','definitions do not prove as-of timing'],
['Standing checks','child absence versus null and raw joins','must remain separate and applicant-level'],
['Underwriter skepticism','association implies policy action','not established'],
['Underwriter skepticism','bureau absence proves no history or underserved status','not established'],
['Underwriter skepticism','historical results transfer to current markets','not established'],
['Underwriter skepticism','group gaps prove causality or fairness','not established']
],columns=['review lens','open item','status']);show(x,'the remaining evidence gaps against the business problem, dictionary, standing checks, and underwriting skepticism.')

,review lens,open item,status
0,Business Problem Statement,"held-out AUC, calibration, selection and risk ...","required in validation, not EDA"
1,Business Problem Statement,financial contribution and adverse sensitivity,requires Finance inputs
2,Dictionary,record-level temporal provenance,definitions do not prove as-of timing
3,Standing checks,child absence versus null and raw joins,must remain separate and applicant-level
4,Underwriter skepticism,association implies policy action,not established
5,Underwriter skepticism,bureau absence proves no history or underserve...,not established
6,Underwriter skepticism,historical results transfer to current markets,not established
7,Underwriter skepticism,group gaps prove causality or fairness,not established


What the output shows: the remaining evidence gaps against the business problem, dictionary, standing checks, and underwriting skepticism.


# Remediations applied

These are reversible analysis-layer treatments. Raw source CSVs remain unchanged; every transformation preserves either the original field or an explicit flag.

In [2]:
# Application-level corrections that are supported by the dictionary and observed data.
def clean_application_for_eda(frame, upper_bounds):
    cleaned = frame.copy()

    # Dictionary contradiction: 365243 cannot be a days-before-application value.
    employed_sentinel = cleaned['DAYS_EMPLOYED'].eq(365243)
    cleaned['DAYS_EMPLOYED_SENTINEL_FLAG'] = employed_sentinel.astype('int8')
    cleaned.loc[employed_sentinel, 'DAYS_EMPLOYED'] = np.nan

    # XNA is retained as a data-quality signal but not treated as verified gender.
    gender_unknown = cleaned['CODE_GENDER'].eq('XNA')
    cleaned['CODE_GENDER_UNKNOWN_FLAG'] = gender_unknown.astype('int8')
    cleaned.loc[gender_unknown, 'CODE_GENDER'] = np.nan

    # Positive values contradict the documented before-application direction.
    for column in ['DAYS_REGISTRATION', 'DAYS_ID_PUBLISH', 'DAYS_LAST_PHONE_CHANGE']:
        invalid = cleaned[column].gt(0)
        cleaned[f'{column}_POSITIVE_FLAG'] = invalid.astype('int8')
        cleaned.loc[invalid, column] = np.nan

    # Do not alter unverified but plausible extremes: flag and use a stable log view.
    for column, bound in upper_bounds.items():
        cleaned[f'{column}_EXTREME_FLAG'] = cleaned[column].gt(bound).astype('int8')
    for column in ['AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE']:
        cleaned[f'{column}_LOG1P'] = np.log1p(cleaned[column])
    return cleaned

# Bounds are learned from the labelled development data only and are flags, not caps.
flag_columns = ['AMT_INCOME_TOTAL', 'AMT_REQ_CREDIT_BUREAU_QRT', 'OBS_30_CNT_SOCIAL_CIRCLE',
                'DEF_30_CNT_SOCIAL_CIRCLE', 'CNT_CHILDREN', 'CNT_FAM_MEMBERS', 'OWN_CAR_AGE']
upper_bounds = train[flag_columns].quantile(.999).to_dict()
train_clean = clean_application_for_eda(train, upper_bounds)
test_clean = clean_application_for_eda(test, upper_bounds)

summary = pd.DataFrame({
    'train_rows_flagged': [int(train_clean['DAYS_EMPLOYED_SENTINEL_FLAG'].sum()),
                           int(train_clean['CODE_GENDER_UNKNOWN_FLAG'].sum()),
                           int(train_clean.filter(like='_EXTREME_FLAG').any(axis=1).sum())],
    'test_rows_flagged': [int(test_clean['DAYS_EMPLOYED_SENTINEL_FLAG'].sum()),
                          int(test_clean['CODE_GENDER_UNKNOWN_FLAG'].sum()),
                          int(test_clean.filter(like='_EXTREME_FLAG').any(axis=1).sum())]
}, index=['employment sentinel recoded to missing', 'unverified gender recoded to missing', 'one or more extreme-value flags'])
show(summary, 'the counts of reversible application-layer corrections and flags; no raw source value was overwritten.')

,train_rows_flagged,test_rows_flagged
employment sentinel recoded to missing,55374,9274
unverified gender recoded to missing,4,0
one or more extreme-value flags,802,126


What the output shows: the counts of reversible application-layer corrections and flags; no raw source value was overwritten.


In [3]:
# Enforced history eligibility: only records demonstrably earlier than the current application are retained.
# bureau_balance was read in chunks, filtered to MONTHS_BALANCE < 0, and linked only through a retained bureau ID.
# The merge below drops balance records with no resolving bureau/application path from candidate features.
history_eligibility = pd.DataFrame([
    ['bureau', len(bureau_raw), len(bureau_pre), 'SK_ID_CURR; DAYS_CREDIT < 0'],
    ['previous_application', len(prev_raw), len(prev_pre), 'SK_ID_CURR; DAYS_DECISION < 0'],
    ['bureau_balance linked', int(bb_link.shape[0]), int(bb_link.bureau_balance_severe_dpd.notna().sum()), 'SK_ID_BUREAU -> bureau -> SK_ID_CURR; MONTHS_BALANCE < 0'],
], columns=['source', 'rows_before_rule', 'rows_eligible_or_linked', 'enforced rule'])
show(history_eligibility, 'the historical records retained or linked after enforcing the approved conservative timing and key-resolution rules.')

,source,rows_before_rule,rows_eligible_or_linked,enforced rule
0,bureau,1716428,1716403,SK_ID_CURR; DAYS_CREDIT < 0
1,previous_application,1670214,1670214,SK_ID_CURR; DAYS_DECISION < 0
2,bureau_balance linked,1716403,769170,SK_ID_BUREAU -> bureau -> SK_ID_CURR; MONTHS_B...


What the output shows: the historical records retained or linked after enforcing the approved conservative timing and key-resolution rules.


## Deliberately not “fixed”

Income, inquiry, social-circle, household, and car-age extremes remain in the source values because this extract cannot prove they are errors. They now carry reproducible flags; any cap, deletion, or manual correction requires source verification and should be fitted only within a training fold during modeling.